# Train your own masked-diffusion language model

Convert an autoregressive instruction model into a masked-diffusion model on a single GPU. **Quick demo** offers three families with small compatible text-only checkpoints and rank-128 LoRA. **Paper reproduction** offers all four original 7B–9B models and uses the rank-1024, 25,000-update configuration.

## 1. Install the training code
This downloads the clean reproduction repository and installs its CUDA dependencies.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/RuurdKuiper/BYOD.git"
REPOSITORY_DIR = Path("/content/BYOD")

if not (REPOSITORY_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
# Colab may ship an old optional TorchAO that conflicts with current PEFT.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("Installation complete.")

## 2. Connect Hugging Face
Add `HF_TOKEN` under Colab **Secrets** or sign in when prompted. The token only needs read access. You must first accept the selected model's license on Hugging Face.

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

## 3. Choose a model and run mode
Start with **Quick demo**. Select **Paper reproduction** only on a high-memory GPU and when you intend to run the complete experiment. Ministral is available only in paper mode because there is no comparable official small text-only Ministral checkpoint.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

QUICK_MODELS = [
    ("Llama 3.2 1B Instruct", "llama"),
    ("Gemma 3 1B IT", "gemma"),
    ("Qwen2.5 1.5B Instruct", "qwen"),
]
PAPER_MODELS = [
    ("Llama 3.1 8B Instruct", "llama"),
    ("Gemma 2 9B IT", "gemma"),
    ("Qwen2.5 7B Instruct", "qwen"),
    ("Ministral 8B Instruct 2410 (text-only)", "ministral"),
]

mode = widgets.ToggleButtons(
    options=[("Quick demo", "quick"), ("Paper reproduction", "paper")],
    value="quick",
    description="Run mode:",
    style={"description_width": "initial"},
)
model = widgets.Dropdown(
    options=QUICK_MODELS,
    value="llama",
    description="Base model:",
    style={"description_width": "initial"},
)

def update_model_options(change):
    previous = model.value
    options = QUICK_MODELS if change["new"] == "quick" else PAPER_MODELS
    model.options = options
    available = {value for _, value in options}
    model.value = previous if previous in available else "llama"

mode.observe(update_model_options, names="value")
save_to_drive = widgets.Checkbox(value=True, description="Save checkpoints to Google Drive")
display(mode, model, save_to_drive)

## 4. Train
The display refreshes with loss curves and recent messages. Dataset/model preparation happens before the first loss point appears. Checkpoints and metrics are written throughout the run.

In [ ]:
import datetime
import subprocess
import sys
from pathlib import Path
import torch

from diffusion_lm.notebook import run_with_dashboard
from train import load_training_config

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime under Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

if save_to_drive.value:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/BYOD-runs")
else:
    output_root = Path("/content/BYOD-runs")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = output_root / f"{model.value}-{mode.value}-{stamp}"
config = load_training_config(model.value, mode.value, output_dir=str(run_dir))
print("Selected model:", config["display_name"])
if mode.value == "paper":
    print("Paper mode selected: 25,000 updates with the exact rank-1024 recipe.")

command = [
    sys.executable, "train.py",
    "--model", model.value,
    "--mode", mode.value,
    "--output-dir", str(run_dir),
]
run_with_dashboard(command, run_dir, config["max_updates"], refresh_seconds=5)

## 5. Inspect the result
`best/` is the inference-ready adapter selected using validation loss. The resolved configuration records exactly what was run.

In [ ]:
import json

print("Run directory:", run_dir)
print("Best adapter:", run_dir / "best")
if (run_dir / "test_metrics.json").exists():
    print(json.dumps(json.loads((run_dir / "test_metrics.json").read_text()), indent=2))